In [1]:
import os
import shutil
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ================= 配置 =================
# [修改] D 通道专用目录
DATASET_ROOT = "./dataset_d_channel_only" 
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")

CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]

# 真实数据映射
FILE_MAPPING = {
    "E:/Project/AMR-Benchmark/PSK8-03-24-25-10h54m48s14.iq": "8PSK",
    "E:/Project/AMR-Benchmark/QAM16-03-24-25-10h52m09s762.iq": "16QAM",
    "E:/Project/AMR-Benchmark/QAM64-03-24-25-10h51m51s805.iq": "64QAM",
    "E:/Project/AMR-Benchmark/ASK2-03-24-25-10h56m09s501.iq": "2ASK",
    "E:/Project/AMR-Benchmark/BPSK-03-24-25-10h52m32s937.iq": "BPSK",
    "E:/Project/AMR-Benchmark/QPSK-03-24-25-10h53m20s46.iq": "QPSK",
    "E:/Project/AMR-Benchmark/FSK2DEV625KHZ-03-24-25-11h08m50s462.iq": "FSK2",
    "E:/Project/AMR-Benchmark/OQPSK-03-24-25-10h54m02s6.iq": "OQPSK",
    "E:/Project/AMR-Benchmark/PI4DQPSK-03-24-25-10h54m23s982.iq": "PI4DQPSK"
}

SAMPLES_PER_IMAGE = 10240
IMAGE_SIZE = 64
TOTAL_SIM_SAMPLES_PER_CLASS = 200 # 演示用，生成少量即可
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# ================= 1. D 通道计算核心 =================

def fast_hist2d(x, y, bins=64, range_bounds=((-1, 1), (-1, 1))):
    x_min, x_max = range_bounds[0]
    y_min, y_max = range_bounds[1]
    x = torch.clamp(x, x_min, x_max)
    y = torch.clamp(y, y_min, y_max)
    x_idx = ((x - x_min) / (x_max - x_min) * (bins - 1)).long()
    y_idx = ((y - y_min) / (y_max - y_min) * (bins - 1)).long()
    flat_idx = y_idx * bins + x_idx
    hist = torch.bincount(flat_idx, minlength=bins*bins)
    return hist.view(bins, bins).float()

def compute_d_channel(iq_complex_data, img_size=64):
    """
    计算 D 通道：差分星座图 (Differential Constellation)
    Formula: D[t] = S[t] * conj(S[t-1])
    Output: (1, H, W)
    """
    if iq_complex_data.dim() == 1:
        iq_complex_data = iq_complex_data.unsqueeze(0)
    
    iq = iq_complex_data.squeeze()
    if iq.numel() < 100: return torch.zeros(1, img_size, img_size)

    # 1. 归一化 (保持原比例)
    scale = torch.sqrt(torch.mean(iq.real**2 + iq.imag**2)) + 1e-9
    iq = iq / (scale) 
    
    # 2. 计算差分信号 (Differential Signal)
    # diff = S[t] * conj(S[t-1])
    # 这步操作会自动提取“相位跳变”信息，并且抵消掉“整体旋转”
    diff_signal = iq[1:] * torch.conj(iq[:-1])
    
    # 差分信号的幅度也会变大 (大约是原来的平方)，需要重新归一化
    # 但为了保持相对跳变的几何结构，我们这里再次鲁棒归一化
    d_scale = torch.sqrt(torch.mean(diff_signal.real**2 + diff_signal.imag**2)) + 1e-9
    diff_signal = diff_signal / (3.0 * d_scale)
    
    real = diff_signal.real
    imag = diff_signal.imag
    
    # 3. 生成直方图 (参数与 R 通道一致)
    bins = img_size
    hist_d = fast_hist2d(real, imag, bins=bins, range_bounds=((-1.2, 1.2), (-1.2, 1.2)))
    
    # 4. 对数处理
    def process(h):
        h = torch.log1p(h)
        return (h - h.min()) / (h.max() - h.min() + 1e-8)
        
    img_tensor = process(hist_d).unsqueeze(0)
    return torch.flip(img_tensor, [1])

def save_tensor_as_grayscale(tensor, path):
    tensor = tensor.detach().cpu().squeeze()
    arr = (tensor.numpy() * 255).astype(np.uint8)
    Image.fromarray(arr, mode='L').save(path)

# ================= 2. 信号仿真 (保持不变) =================
# ... (此处省略 generate_sim_signal 等长代码，与之前完全一致，直接调用即可) ...
# 为了完整性，我还是把关键的仿真部分贴上，确保你能直接运行

def generate_rrc_filter(N_taps, beta, sps):
    t = torch.arange(-(N_taps - 1) // 2, (N_taps - 1) // 2 + 1, dtype=torch.float32) / sps
    mid_idx = (N_taps - 1) // 2
    t[mid_idx] = 1e-8
    num = torch.sin(np.pi * t * (1 - beta)) + 4 * beta * t * torch.cos(np.pi * t * (1 + beta))
    den = np.pi * t * (1 - (4 * beta * t)**2)
    coeffs = num / den
    coeffs[mid_idx] = 1 - beta + (4 * beta / np.pi)
    return coeffs / torch.sqrt(torch.sum(coeffs**2))

def apply_channel_effects(sig, snr_db):
    N = len(sig)
    freq_offset = np.random.uniform(-50, 50) 
    t = torch.arange(N, dtype=torch.float32) / 80e3
    phase_offset = np.random.uniform(0, 2*np.pi)
    rotation = torch.exp(1j * (2 * np.pi * freq_offset * t + phase_offset))
    sig = sig * rotation
    phase_jitter = torch.randn(N) * 0.005 
    sig = sig * torch.exp(1j * phase_jitter)
    g_imbalance = np.random.uniform(0.98, 1.02)
    p_imbalance = np.random.uniform(-1, 1) * np.pi/180
    real = sig.real * g_imbalance
    imag = sig.imag * np.cos(p_imbalance) - sig.real * np.sin(p_imbalance)
    sig = torch.complex(real, imag)
    sig_power = torch.mean(torch.abs(sig)**2)
    real_snr = snr_db + 5 
    noise_power = sig_power / (10**(real_snr/10.0))
    noise = (torch.randn_like(sig) + 1j*torch.randn_like(sig)) * torch.sqrt(noise_power/2)
    return sig + noise

# 辅助函数: 32QAM 和 APSK
def get_32qam_constellation():
    points = []
    for x in [-5, -3, -1, 1, 3, 5]:
        for y in [-5, -3, -1, 1, 3, 5]:
            if not (abs(x) == 5 and abs(y) == 5): 
                points.append(x + 1j*y)
    return torch.tensor(points, dtype=torch.complex64)

def get_apsk_symbols(M, ratios, num_symbols):
    if M == 16:
        R1, R2 = 1.0, ratios[0]
        pts_inner = R1 * torch.exp(1j * (torch.arange(4)*2*np.pi/4 + np.pi/4))
        pts_outer = R2 * torch.exp(1j * (torch.arange(12)*2*np.pi/12))
        constellation = torch.cat([pts_inner, pts_outer])
    elif M == 32:
        R1, R2, R3 = 1.0, ratios[0], ratios[1]
        pts_inner = R1 * torch.exp(1j * (torch.arange(4)*2*np.pi/4 + np.pi/4))
        pts_mid = R2 * torch.exp(1j * (torch.arange(12)*2*np.pi/12))
        pts_outer = R3 * torch.exp(1j * (torch.arange(16)*2*np.pi/16))
        constellation = torch.cat([pts_inner, pts_mid, pts_outer])
    indices = torch.randint(0, M, (num_symbols,))
    return constellation[indices]

def generate_sim_signal(mod_type, N, snr_db):
    Fs, Rs = 80e3, 20e3
    sps = int(Fs / Rs)
    num_symbols = int(N / sps) + 50
    
    syms = None
    if mod_type == "BPSK":
        syms = 2*torch.randint(0, 2, (num_symbols,)) - 1 + 0j
    elif mod_type == "QPSK":
        idx = torch.randint(0, 4, (num_symbols,))
        syms = torch.exp(1j * (np.pi/4 + idx * np.pi/2))
    elif mod_type == "8PSK":
        idx = torch.randint(0, 8, (num_symbols,))
        syms = torch.exp(1j * idx * 2*np.pi/8)
    elif mod_type == "OQPSK":
        idx = torch.randint(0, 4, (num_symbols,))
        syms = torch.exp(1j * (np.pi/4 + idx * np.pi/2))
    elif mod_type == "PI4DQPSK":
        bits = torch.randint(0, 4, (num_symbols,))
        phase_shifts = torch.tensor([np.pi/4, 3*np.pi/4, -np.pi/4, -3*np.pi/4])[bits]
        phases = torch.cumsum(phase_shifts, dim=0)
        syms = torch.exp(1j * phases)
    elif mod_type == "2ASK":
        bits = torch.randint(0, 2, (num_symbols,)).float()
        syms = bits + 0j
    elif mod_type == "16QAM":
        idx = torch.randint(0, 16, (num_symbols,))
        i = (idx // 4) * 2 - 3
        q = (idx % 4) * 2 - 3
        syms = i + 1j*q
    elif mod_type == "32QAM":
        constellation = get_32qam_constellation()
        idx = torch.randint(0, 32, (num_symbols,))
        syms = constellation[idx]
    elif mod_type == "64QAM":
        idx = torch.randint(0, 64, (num_symbols,))
        i = (idx // 8) * 2 - 7
        q = (idx % 8) * 2 - 7
        syms = i + 1j*q
    elif mod_type == "16APSK":
        syms = get_apsk_symbols(16, [2.7], num_symbols)
    elif mod_type == "32APSK":
        syms = get_apsk_symbols(32, [2.6, 4.5], num_symbols)
    elif mod_type == "FSK2":
        bits = torch.randint(0, 2, (N + 100,))
        f_dev = np.random.uniform(500, 6000) 
        freqs = (bits * 2 - 1) * f_dev
        phase = 2 * np.pi * torch.cumsum(freqs/Fs, dim=0)
        sig = torch.exp(1j * phase)[:N]
        return apply_channel_effects(sig, snr_db)

    upsampled = torch.zeros(num_symbols * sps, dtype=torch.complex64)
    if mod_type == "OQPSK":
        upsampled_i = torch.zeros(num_symbols * sps)
        upsampled_q = torch.zeros(num_symbols * sps)
        upsampled_i[::sps] = syms.real
        q_temp = torch.zeros(num_symbols * sps)
        q_temp[::sps] = syms.imag
        upsampled_q = torch.roll(q_temp, shifts=sps//2)
        upsampled_q[:sps//2] = 0
        upsampled = upsampled_i + 1j * upsampled_q
    else:
        upsampled[::sps] = syms

    taps = 10 * sps + 1
    h = generate_rrc_filter(taps, 0.35, sps)
    real = F.conv1d(upsampled.real.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    imag = F.conv1d(upsampled.imag.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    sig = (real + 1j*imag).squeeze()
    
    if len(sig) > N: sig = sig[:N]
    else: sig = F.pad(sig, (0, N - len(sig)))
    return apply_channel_effects(sig, snr_db)

# ================= 3. 执行生成 =================

if __name__ == "__main__":
    if os.path.exists(DATASET_ROOT):
        shutil.rmtree(DATASET_ROOT)
    
    print("===== Generating D-Channel (Differential Constellation) =====")
    for cls_name in CLASS_NAMES:
        os.makedirs(os.path.join(SIM_TRAIN_DIR, cls_name), exist_ok=True)
        print(f"Generating {cls_name}...")
        
        # 为了演示，只生成高信噪比的，让你看清楚形状
        snr = 20 
        for i in range(10): 
            sig = generate_sim_signal(cls_name, SAMPLES_PER_IMAGE, snr)
            heatmap = compute_d_channel(sig.to(DEVICE), IMAGE_SIZE)
            save_path = os.path.join(SIM_TRAIN_DIR, cls_name, f"{cls_name}_{i:04d}.png")
            save_tensor_as_grayscale(heatmap, save_path)
    
    print("生成完毕。请打开 dataset_d_channel_only/sim_train 文件夹")
    print("对比 8PSK (8个点) 和 PI4DQPSK (4个点 X形) 的区别。")

d:\anaconda3\envs\pytorch1_12_1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


===== Generating D-Channel (Differential Constellation) =====
Generating BPSK...
Generating QPSK...
Generating 8PSK...
Generating OQPSK...
Generating PI4DQPSK...
Generating 16QAM...
Generating 32QAM...
Generating 64QAM...
Generating 2ASK...


C:\Users\CPC-ML\AppData\Local\Temp\ipykernel_6680\1386389145.py:187: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  i = (idx // 4) * 2 - 3
C:\Users\CPC-ML\AppData\Local\Temp\ipykernel_6680\1386389145.py:196: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  i = (idx // 8) * 2 - 7


Generating 16APSK...
Generating 32APSK...
Generating FSK2...
生成完毕。请打开 dataset_d_channel_only/sim_train 文件夹
对比 8PSK (8个点) 和 PI4DQPSK (4个点 X形) 的区别。
